<アイデア>

自分で作った最小全域木をクエリによって更新する

<アルゴリズム>

- プリム法で初期グループとその全域木を作る
  - （TODO）密度が高い点かつ、グループ数が多いとこから全域木を作るべき
- 3 <= グループサイズ <= L となるグループはクエリを 1 回行うだけ
- グループサイズ > L となるグループについて

  - 全点の訪問回数を 0 に初期化
  - 現在の全域木から適当な辺を削除して、連結成分を 2 つ得る
  - どちらかの連結成分が L 以下ならその成分をクエリ対象にいれる
  - たくさんクエリ対象になる点で、かつ訪問回数がすくない成分を対象にしてクエリを行う
    - アイディア 1：目標訪問回数 - 現在の訪問回数の合計で競う
    - アイディア 2：純粋に点数 - 訪問回数の合計でソート
  - クエリを行う
    - クエリの回答と、削除した辺、元の連結成分を結合する
  - 上記を一定繰り返す

- クエリ予定回数 + 0.01 / グループ点数で初期化
- 最もコストの小さいグループにクエリ回数を 1 回加算して上げる
- クエリが 400 回になるまで繰り返す

<評価>
avg: 309964.9824460464


In [ ]:
IS_ONLINE_JUDGE = False

# =================================
import time

GLOBAL_START_TIME = time.perf_counter()


def debug_print(*args, **kwargs):
    if IS_ONLINE_JUDGE:
        return
    print(*args, **kwargs)


def print_elapsed_time():
    debug_print(f"elapsed time: {(time.perf_counter() - GLOBAL_START_TIME) * 1000:.2f}msecs")


# =================================
import heapq
import math
import random


INF = 10**18
FILE_NUM = 100


# =================================
def calc_rectangle_area(rectangle):
    l, r, t, b = rectangle
    area = (r - l) * (b - t)
    return area


def calc_dist(p1: tuple, p2: tuple) -> float:
    return math.dist(p1, p2)


def prim(
    graph: dict[int, list[tuple[int, int]]],
) -> tuple[list[tuple[int, int]], list[int]]:
    """
    G: 隣接グラフ
    G := [ [(v_0, cost_0), (v_1,cost_1),..], [(v_2, cost_2)],...]

    返り値 ans :最小全域木の重みの総和
    """
    v_list = list(graph.keys())
    used = {v: False for v in v_list}

    init_v = v_list[0]  # 適当な点を選ぶ
    used[init_v] = True
    que = [(cost, init_v, v) for v, cost in graph[init_v]]
    heapq.heapify(que)

    ans_v: list[int] = [init_v]
    ans_edges: list[tuple[int, int]] = []
    while que:
        cost_v, from_v, to_v = heapq.heappop(que)
        if used[to_v]:
            continue
        used[to_v] = True
        ans_edges.append((min(from_v, to_v), max(from_v, to_v)))
        ans_v.append(to_v)
        for nxt, cost_nxt in graph[to_v]:
            if used[nxt]:
                continue
            heapq.heappush(que, (cost_nxt, to_v, nxt))
    return ans_edges, ans_v


def prim_k(graph: list[list[float]], init_v: int, k: int, used_v: set[int], N):
    """
    init_vを含むk頂点の最小全域木を求める
    """
    used = used_v | {init_v}
    used.add(init_v)
    que = [(graph[init_v][v], init_v, v) for v in range(N)]
    heapq.heapify(que)

    ans_v = [init_v]
    ans_edges: list[tuple[int, int]] = []
    ans_cost = 0.0
    while que and len(ans_v) < k:
        cost_v, from_v, to_v = heapq.heappop(que)
        if to_v in used:
            continue
        used.add(to_v)
        ans_edges.append((min(from_v, to_v), max(from_v, to_v)))
        ans_v.append(to_v)
        ans_cost += cost_v
        for nxt in range(N):
            if nxt in used:
                continue
            heapq.heappush(que, (graph[to_v][nxt], to_v, nxt))
    return ans_edges, ans_v, ans_cost


# =================================


class EnvOffline:
    def __init__(self, input_file_path: str, output_file_path: str):
        with open(input_file_path) as f:
            lines = f.readlines()
            N, M, Q, L, W = map(int, lines[0].split())
            G = list(map(int, lines[1].split()))
            rectangles = []
            for l in range(2, 2 + N):
                rectangles.append(list(map(int, lines[l].split())))
            coordinates = []
            for l in range(2 + N, 2 + N + N):
                coordinates.append(tuple(map(int, lines[l].split())))

        self.N, self.M, self.Q, self.L, self.W = N, M, Q, L, W
        self.G = G
        self.rectangles = rectangles
        self.coordinates = coordinates
        self.cneter_points = [((l + r) / 2, (c + d) / 2) for l, r, c, d in rectangles]
        self._query_history: list[str] = []

        self.input_file_path = input_file_path
        self.output_file_path = output_file_path

    def query(self, c_list: list[int]) -> list[tuple[int, int]]:
        query_str = " ".join(["?", str(len(c_list)), *map(str, c_list)])
        self._query_history.append(query_str)

        now_graph: dict[int, list] = {v: [] for v in c_list}
        for c1 in c_list:
            for c2 in c_list:
                if c1 == c2:
                    continue
                x1, y1 = self.coordinates[c1]
                x2, y2 = self.coordinates[c2]
                dist = abs(x1 - x2) ** 2 + abs(y1 - y2) ** 2
                now_graph[c1].append((c2, dist))
                now_graph[c2].append((c1, dist))

        ans_edges, ans_v = prim(now_graph)
        return ans_edges

    def answer(
        self,
        groups: list[list[int]],
        edges: list[list[tuple[int, int]]],
    ):
        cost = 0.0
        ans_str = "!\n"
        for i in range(len(groups)):
            ans_str += " ".join(map(str, groups[i])) + "\n"
            for e in edges[i]:
                ans_str += " ".join(map(str, e)) + "\n"
                dist = calc_dist(self.coordinates[e[0]], self.coordinates[e[1]])
                cost += dist

        for query_str in self._query_history:
            ans_str += query_str + "\n"

        with open(self.output_file_path, "w") as f:
            f.write(ans_str)

        return cost


class EnvOnline:
    def __init__(self):
        N, M, Q, L, W = map(int, input().split())
        G = list(map(int, input().split()))
        rectangles = []
        for l in range(N):
            rectangles.append(list(map(int, input().split())))

        self.N, self.M, self.Q, self.L, self.W = N, M, Q, L, W
        self.G = G
        self.rectangles = rectangles
        self.cneter_points = [((l + r) / 2, (c + d) / 2) for l, r, c, d in rectangles]

    def query(self, c_list: list[int]) -> list[tuple[int, int]]:
        query_str = " ".join(["?", str(len(c_list)), *map(str, c_list)])
        print(query_str)
        return [tuple(map(int, input().split())) for _ in range(len(c_list) - 1)]

    def answer(
        self,
        groups: list[list[int]],
        edges: list[list[tuple[int, int]]],
    ):
        ans_str = "!\n"
        for i in range(len(groups)):
            ans_str += " ".join(map(str, groups[i])) + "\n"
            for e in edges[i]:
                ans_str += " ".join(map(str, e)) + "\n"
        print(ans_str)

<アイデア>

自分で作った最小全域木をクエリによって更新する

<アルゴリズム>

- プリム法で初期グループとその全域木を作る
  - （TODO）密度が高い点かつ、グループ数が多いとこから全域木を作るべき
- 3 <= グループサイズ <= L となるグループはクエリを 1 回行うだけ
- グループサイズ > L となるグループについて

  - 全点の訪問回数を 0 に初期化
  - 現在の全域木から適当な辺を削除して、連結成分を 2 つ得る
  - どちらかの連結成分が L 以下ならその成分をクエリ対象にいれる
  - たくさんクエリ対象になる点で、かつ訪問回数がすくない成分を対象にしてクエリを行う
    - アイディア 1：目標訪問回数 - 現在の訪問回数の合計で競う
    - アイディア 2：純粋に点数 - 訪問回数の合計でソート
  - クエリを行う
    - クエリの回答と、削除した辺、元の連結成分を結合する
  - 上記を一定繰り返す

- クエリ予定回数 + 0.01 / グループ点数で初期化
- 最もコストの小さいグループにクエリ回数を 1 回加算して上げる
- クエリが 400 回になるまで繰り返す


In [104]:
def calc_greedy_answer(env: EnvOffline, target_points):
    graph: list[list[float]] = [[0 for _ in range(env.N)] for _ in range(env.N)]
    for i in range(env.N):
        for j in range(env.N):
            if i == j:
                continue
            dist = calc_dist(target_points[i], target_points[j])
            graph[i][j] = dist
            graph[j][i] = dist

    groups = [(i, g) for i, g in enumerate(env.G)]
    groups = sorted(groups, key=lambda x: x[1], reverse=True)

    ans_edges = [None for _ in range(len(groups))]
    ans_v = [None for _ in range(len(groups))]

    now_used: set = set()
    not_used: set = set(range(env.N))
    for group_n, group_size in groups:
        v = not_used.pop()
        prim_edges, prim_v, _ = prim_k(graph, v, group_size, now_used, env.N)

        ans_edges[group_n] = prim_edges
        ans_v[group_n] = prim_v
        now_used.update(prim_v)
        not_used.difference_update(prim_v)

    return ans_v, ans_edges

In [105]:
from collections import defaultdict, deque


class MyGraph:
    def __init__(self, vs, edges, group_ind):
        self.vs = vs
        self.edges = edges
        self.group_ind = group_ind

        self.visited_cnt = {v: 0 for v in vs}

    def update_edges(self, qv, qvs, remove_edge, query_edges):
        qvs = set(qvs)
        for v in qvs:
            self.visited_cnt[v] += 1

        # qvs から伸びる辺を削除
        del_edges = []
        for i, e in enumerate(self.edges):
            if e[0] in qvs or e[1] in qvs:
                continue
            del_edges.append(e)
        self.edges = del_edges + [remove_edge] + query_edges

    def calc_select_score(self, vs):
        cost = 0
        cost += len(vs)
        cost -= sum(self.visited_cnt[v] for v in vs)
        return cost

    def cut_graph(self, edge_id):
        graph = defaultdict(list)
        for i, e in enumerate(self.edges):
            if i == edge_id:
                continue
            graph[e[0]].append(e[1])
            graph[e[1]].append(e[0])

        target_edge = self.edges[edge_id]
        v1, v2 = target_edge

        group1 = self.bfs(graph, v1)
        group2 = self.bfs(graph, v2)
        return v1, v2, group1, group2

    def bfs(self, graph, init_v):
        visited = set([init_v])
        q = deque([init_v])
        while q:
            node = q.pop()
            for nv in graph[node]:
                if nv not in visited:
                    visited.add(nv)
                    q.append(nv)
        return visited

In [ ]:
def update_mst(env: EnvOnline, ans_v, ans_edges):
    group_query_cnt = [0 for _ in range(env.M)]
    group_query_v_cnt = [0 for _ in range(env.M)]
    query_cnt = 0

    largest_groups = []
    for n in range(env.M):
        if 3 <= len(ans_v[n]):
            group_query_cnt[n] = 1
            group_query_v_cnt[n] = env.L
            query_cnt += 1
            if env.L < len(ans_v[n]):
                largest_groups.append(n)

    INF = 10**18
    for _ in range(400 - query_cnt):
        min_cost = INF
        target_group = None
        for g in largest_groups:
            cost = group_query_v_cnt[g] / len(ans_v[g])
            if cost < min_cost:
                min_cost = cost
                target_group = g
        if target_group:
            group_query_v_cnt[target_group] += env.L
            group_query_cnt[target_group] += 1
            query_cnt += 1

    for g in range(env.M):
        now_v = ans_v[g]
        now_edges = ans_edges[g]
        if len(now_v) <= 3:
            continue
        if 3 <= len(now_v) <= env.L:
            ans_edges[g] = env.query(now_edges)

        my_graph = MyGraph(now_v, now_edges, g)
        now_score = -INF
        now_target = None
        for _ in range(group_query_cnt[g]):
            for ei in range(len(now_edges)):
                v1, v2, group1, group2 = my_graph.cut_graph(ei)
                if 3 <= len(group1) <= env.L:
                    score = my_graph.calc_select_score(group1)
                    if score > now_score:
                        now_score = score
                        now_target = (v1, v2, group1, group2)
                if 3 <= len(group2) <= env.L:
                    score = my_graph.calc_select_score(group2)
                    if score > now_score:
                        now_score = score
                        now_target = (v2, v1, group2, group1)
            if now_target is None:
                break

            qv, ov, q_group, other_gruop = now_target
            new_edges = env.query(q_group)
            my_graph.update_edges(qv, q_group, (qv, ov), new_edges)

        ans_edges[g] = my_graph.edges

    return ans_v, ans_edges


def solve(env: EnvOffline):
    ans_v, ans_edges = calc_greedy_answer(env, env.cneter_points)
    ans_v, ans_edges = update_mst(env, ans_v, ans_edges)
    return ans_v, ans_edges

In [107]:
file_num = 0
input_file_path = f"../in/{file_num:04d}.txt"
output_file_path = f"../out/{file_num:04d}.txt"
env = EnvOffline(input_file_path, output_file_path)
ans_v, ans_edges = solve(env)
env.answer(ans_v, ans_edges)

329470.9782381991
322467.95932073053


322467.95932073053